# Step 1 — Explore and prepare the data

We want to predict a house's **SalePrice** from its characteristics. Run the cells from top to bottom. Each short explanation says **why** the next code is needed.

At the end of this step, the data will be ready for models. We will train the models in a later step.

## 1. Load the file
`df` is a Pandas table. Each row is one house; each column describes something about it.

In [1]:
import pandas as pd


df = pd.read_csv('../data/raw/House_Prices.csv')
print('Houses:', len(df))
print('Columns:', len(df.columns))
display(df[['Id', 'GrLivArea', 'OverallQual', 'Neighborhood', 'SalePrice']].head())

Houses: 2919
Columns: 81


,Id,GrLivArea,OverallQual,Neighborhood,SalePrice
0,1,1710,7,CollgCr,208500.0
1,2,1262,6,Veenker,181500.0
2,3,1786,7,CollgCr,223500.0
3,4,1717,7,Crawfor,140000.0
4,5,2198,8,NoRidge,250000.0


## 2. Check which prices are safe to learn from
The first 1,460 prices are whole numbers. Every later price has decimals. This suggests the later prices might be generated predictions rather than real sales. We cannot prove that from this file alone. We will use the first 1,460 rows for training **until the source of the later prices is confirmed**.

In [2]:
sales = df[df['Id'] <= 1460].copy()
later = df[df['Id'] > 1460]

print('Rows used for learning:', len(sales))
print('Later rows set aside:', len(later))
display(df.loc[df['Id'].isin([1460, 1461]), ['Id', 'SalePrice']])

Rows used for learning: 1460
Later rows set aside: 1459


,Id,SalePrice
1459,1460,147500.000000
1460,1461,169277.052498


## 3. Understand column types
A **numeric** column contains a number, like living area. A **categorical** column contains a label, like neighborhood. Different types need different preparation.

In [3]:
print('Table shape:', sales.shape)
sales.info()

numeric_columns = sales.select_dtypes(include='number').columns
text_columns = sales.select_dtypes(exclude='number').columns
print('Numeric columns:', len(numeric_columns))
print('Categorical columns:', len(text_columns))

Table shape: (1460, 81)
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  


## 4. Count missing values and duplicates
An empty `PoolQC` may mean there is no pool. We count empty cells now; later we will fill them without changing the original CSV. A duplicate is a repeated row or `Id`.

In [4]:
missing = sales.isna().sum().sort_values(ascending=False)
print('Columns with missing values:', (missing > 0).sum())
print('Total missing cells:', missing.sum())
display(missing[missing > 0].head(10).to_frame('missing cells'))

print('Duplicate rows:', sales.duplicated().sum())
print('Duplicate Ids:', sales['Id'].duplicated().sum())

Columns with missing values: 19
Total missing cells: 7829


,missing cells
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageQual,81
GarageFinish,81
GarageType,81


Duplicate rows: 0
Duplicate Ids: 0


## 5. Understand `SalePrice`
The **median** is the middle price. The lowest and highest prices may reveal unusual houses. We inspect them; we do not delete them automatically.

In [5]:
display(sales['SalePrice'].describe())
print('Missing prices:', sales['SalePrice'].isna().sum())
print('Prices <= 0:', (sales['SalePrice'] <= 0).sum())

display(sales.nsmallest(5, 'SalePrice')[['Id', 'SalePrice', 'GrLivArea']])
display(sales.nlargest(5, 'SalePrice')[['Id', 'SalePrice', 'GrLivArea']])

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

Missing prices: 0
Prices <= 0: 0


,Id,SalePrice,GrLivArea
495,496,34900.0,720
916,917,35311.0,480
968,969,37900.0,968
533,534,39300.0,334
30,31,40000.0,1317


,Id,SalePrice,GrLivArea
691,692,755000.0,4316
1182,1183,745000.0,4476
1169,1170,625000.0,3627
898,899,611657.0,2364
803,804,582933.0,2822


## 6. Check possible mistakes and outliers
A construction year after the sale year would be inconsistent. Very large houses may be valid, so we only flag them for inspection.

In [6]:
print('Built after sale:', (sales['YearBuilt'] > sales['YrSold']).sum())
print('Renovated after sale:', (sales['YearRemodAdd'] > sales['YrSold']).sum())
print('Living area <= 0:', (sales['GrLivArea'] <= 0).sum())
print('Living area > 4000:', (sales['GrLivArea'] > 4000).sum())

display(sales.loc[sales['YearRemodAdd'] > sales['YrSold'],
                  ['Id', 'YearRemodAdd', 'YrSold']])
display(sales.loc[sales['GrLivArea'] > 4000,
                  ['Id', 'GrLivArea', 'SalePrice']])

Built after sale: 0
Renovated after sale: 1
Living area <= 0: 0
Living area > 4000: 4


,Id,YearRemodAdd,YrSold
523,524,2008,2007


,Id,GrLivArea,SalePrice
523,524,4676,184750.0
691,692,4316,755000.0
1182,1183,4476,745000.0
1298,1299,5642,160000.0


## 7. Choose `X` and `y`
`X` is the house information. `y` is the price we want to predict.

We start with **six understandable columns**: living area, quality, construction year, street frontage, neighborhood, and garage type. This keeps the first model and its future form simple. We can add more columns later if needed. `Id` is a row number, so it is not a house characteristic.

In [7]:
features = [
    'GrLivArea', 'OverallQual', 'YearBuilt',
    'LotFrontage', 'Neighborhood', 'GarageType'
]
X = sales[features].copy()
y = sales['SalePrice'].copy()

print('House columns in X:', list(X.columns))
print('Number of prices in y:', len(y))

House columns in X: ['GrLivArea', 'OverallQual', 'YearBuilt', 'LotFrontage', 'Neighborhood', 'GarageType']
Number of prices in y: 1460


## 8. Split the houses
We keep some houses for learning (**training**) and some for checking the result (**test**). We split **before** filling missing values.

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print('Training houses:', len(X_train))
print('Test houses:', len(X_test))

Training houses: 1168
Test houses: 292


## 9. Fill missing values
Among our six columns, only two have empty cells. We use the training houses to find the middle `LotFrontage`. An empty `GarageType` becomes `Missing`.

In [9]:
frontage = X_train['LotFrontage'].median()
X_train['LotFrontage'] = X_train['LotFrontage'].fillna(frontage)
X_test['LotFrontage'] = X_test['LotFrontage'].fillna(frontage)

X_train['GarageType'] = X_train['GarageType'].fillna('Missing')
X_test['GarageType'] = X_test['GarageType'].fillna('Missing')

## 10. Change names into 0/1 columns
`get_dummies` turns neighborhood and garage names into numbers. The last line gives the test table the **same columns** as the training table.

In [10]:
X_train = pd.get_dummies(X_train, dtype=int)
X_test = pd.get_dummies(X_test, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## 11. Scale the number columns
The four original number columns use different units. The scaler learns from training houses, then applies the same rule to test houses.

In [11]:
from sklearn.preprocessing import StandardScaler

numbers = ['GrLivArea', 'OverallQual', 'YearBuilt', 'LotFrontage']
scaler = StandardScaler()
X_train[numbers] = scaler.fit_transform(X_train[numbers])
X_test[numbers] = scaler.transform(X_test[numbers])

print('Training:', X_train.shape)
print('Test:', X_test.shape)
print('Missing values:', X_train.isna().sum().sum() + X_test.isna().sum().sum())

Training: (1168, 36)
Test: (292, 36)
Missing values: 0


## Step 1 result
- We inspected the data, including types, missing cells, duplicates, price extremes, and possible mistakes.
- We chose six house characteristics for `X` and `SalePrice` for `y`.
- We split the 1,460 provisional labeled houses into training and test houses.
- We filled missing cells, changed category names into 0/1 columns, and scaled number columns. The values used for filling and scaling came from training houses only.

**Next:** make charts in Step 2. For cross-validation later, we will repeat this preparation inside each model's pipeline. The later 1,459 prices still need source confirmation.